In [5]:
import math
import random
import heapq
from dataclasses import dataclass

# =========================
# 1. Simulation Parameters
# =========================

@dataclass
class SimParams:
    P: int  # no. of preparation rooms
    R: int  # no. of recovery rooms
    mean_interarrival: float = 25.0
    mean_prep: float = 40.0
    mean_surg: float = 20.0
    mean_rec: float = 40.0
    warmup: float = 500.0      # warm-up period (time units)
    run_length: float = 1000.0 # observation window (time units)
    twisted: bool = False      # if True, use twisted (more variable) surgery times


def exp_sample(rng: random.Random, mean: float) -> float:
    # Sample from Exp(mean).
    return rng.expovariate(1.0 / mean)


def sample_surg_time(rng: random.Random, params: SimParams) -> float:
    # Surgery time:
    # - Original: Exp(mean = params.mean_surg)
    # - Twisted: mixture distribution with same mean but higher variance.
    # Example: 30% "severe" (mean 40), 70% "mild" (mean ~11.43),
    # giving E[S] ≈ 20.

    if not params.twisted:
        return exp_sample(rng, params.mean_surg)

    u = rng.random()
    if u < 0.3:
        # severe case
        return exp_sample(rng, 40.0)
    else:
        # mild case
        return exp_sample(rng, 11.4285714286)


# ======================
# 2. Single replication
# ======================

def simulate(params: SimParams, seed: int):
    rng = random.Random(seed)
    warmup = params.warmup
    end_time = warmup + params.run_length

    # Current time
    t = 0.0
    event_counter = 0
    event_queue = []  # priority queue of (time, counter, type, data)

    # ---- State variables ----
    # Queue before preparation
    Q_before_prep = 0
    last_Q_change = 0.0
    area_Q = 0.0

    # Preparation
    prep_occupancy = 0               # number of prep rooms occupied
    queue_prep2OR = 0                # prepped patients stuck in prep waiting for OR

    # Time-weighted preparation occupancy
    last_prep_change = 0.0
    area_prep_busy = 0.0

    # Operating room state: 'IDLE', 'BUSY', 'BLOCKED'
    OR_state = 'IDLE'
    last_OR_change = 0.0
    time_OR_busy = 0.0
    time_OR_blocked = 0.0
    time_OR_idle = 0.0

    # Recovery
    rec_occupancy = 0
    last_rec_change = 0.0
    time_rec_full = 0.0  # time all recovery rooms are busy

    # ---- Helpers for time-average statistics ----

    def record_Q_change(new_Q, now):
        nonlocal Q_before_prep, last_Q_change, area_Q
        prev_Q = Q_before_prep
        start = max(last_Q_change, warmup)
        end = min(now, end_time)
        if end > start:
            area_Q += prev_Q * (end - start)
        Q_before_prep = new_Q
        last_Q_change = now

    def record_OR_change(new_state, now):
        nonlocal OR_state, last_OR_change, time_OR_busy, time_OR_blocked, time_OR_idle
        prev_state = OR_state
        start = max(last_OR_change, warmup)
        end = min(now, end_time)
        if end > start:
            dt = end - start
            if prev_state == 'BUSY':
                time_OR_busy += dt
            elif prev_state == 'BLOCKED':
                time_OR_blocked += dt
            elif prev_state == 'IDLE':
                time_OR_idle += dt
        OR_state = new_state
        last_OR_change = now

    def record_rec_change(new_occ, now):
        nonlocal rec_occupancy, last_rec_change, time_rec_full
        prev_occ = rec_occupancy
        start = max(last_rec_change, warmup)
        end = min(now, end_time)
        if end > start:
            if prev_occ == params.R:
                time_rec_full += (end - start)
        rec_occupancy = new_occ
        last_rec_change = now

    def record_prep_change(new_occ, now):
      nonlocal prep_occupancy, last_prep_change, area_prep_busy
      prev_occ = prep_occupancy
      start = max(last_prep_change, warmup)
      end = min(now, end_time)
      if end > start:
          area_prep_busy += prev_occ * (end - start)
      prep_occupancy = new_occ
      last_prep_change = now


    # ---- Event scheduling helper ----

    def schedule(time_, etype, data=None):
        nonlocal event_counter
        if time_ > end_time + 1e-9:
            return  # don't bother scheduling far beyond
        event_counter += 1
        heapq.heappush(event_queue, (time_, event_counter, etype, data))

    # Initial timestamps for state tracking
    last_Q_change = 0.0
    last_OR_change = 0.0
    last_rec_change = 0.0

    # Schedule the 1st arrival
    schedule(exp_sample(rng, params.mean_interarrival), 'ARRIVAL')

    # ===========
    # Event loop
    # ===========
    while event_queue:
        t, _, etype, data = heapq.heappop(event_queue)
        if t > end_time:
            break

        if etype == 'ARRIVAL':
            # Schedule next arrival
            next_arr = t + exp_sample(rng, params.mean_interarrival)
            schedule(next_arr, 'ARRIVAL')

            # Try to enter preparation
            if prep_occupancy < params.P:
                # Directly into prep
                record_prep_change(prep_occupancy + 1, t)
                prep_time = exp_sample(rng, params.mean_prep)
                schedule(t + prep_time, 'PREP_COMPLETE')
            else:
                # Wait in queue before prep
                record_Q_change(Q_before_prep + 1, t)

        elif etype == 'PREP_COMPLETE':
            # Patient has finished preparation
            if OR_state == 'IDLE':
                # Move to OR immediately
                record_prep_change(prep_occupancy - 1, t)

                # Start prep for next in queue, if any
                if Q_before_prep > 0:
                    record_Q_change(Q_before_prep - 1, t)
                    prep_occupancy += 1
                    prep_time = exp_sample(rng, params.mean_prep)
                    schedule(t + prep_time, 'PREP_COMPLETE')

                # Start surgery
                record_OR_change('BUSY', t)
                surg_time = sample_surg_time(rng, params)
                schedule(t + surg_time, 'SURG_COMPLETE')
            else:
                # Patient finished prep but must hold the prep room while waiting for OR
                queue_prep2OR += 1

        elif etype == 'SURG_COMPLETE':
            # Surgery finished: try to move to recovery
            if rec_occupancy < params.R:
                # Move to recovery
                record_rec_change(rec_occupancy + 1, t)
                rec_time = exp_sample(rng, params.mean_rec)
                schedule(t + rec_time, 'REC_COMPLETE')

                # OR becomes idle, then immediately may take next prep-finished patient
                record_OR_change('IDLE', t)
                if queue_prep2OR > 0:
                    queue_prep2OR -= 1
                    prep_occupancy -= 1
                    # Start prep for new patient if queue exists
                    if Q_before_prep > 0:
                        record_Q_change(Q_before_prep - 1, t)
                        prep_occupancy += 1
                        prep_time = exp_sample(rng, params.mean_prep)
                        schedule(t + prep_time, 'PREP_COMPLETE')
                    # Start new surgery
                    record_OR_change('BUSY', t)
                    surg_time = sample_surg_time(rng, params)
                    schedule(t + surg_time, 'SURG_COMPLETE')
                # else: OR remains idle
            else:
                # Recovery is full -> OR becomes blocked
                record_OR_change('BLOCKED', t)

        elif etype == 'REC_COMPLETE':
            # A recovery bed frees
            record_rec_change(rec_occupancy - 1, t)

            if OR_state == 'BLOCKED':
                # Move blocked patient from OR to recovery immediately
                record_rec_change(rec_occupancy + 1, t)
                rec_time = exp_sample(rng, params.mean_rec)
                schedule(t + rec_time, 'REC_COMPLETE')

                # OR now idle; may start new surgery
                record_OR_change('IDLE', t)
                if queue_prep2OR > 0:
                    queue_prep2OR -= 1
                    prep_occupancy -= 1
                    if Q_before_prep > 0:
                        record_Q_change(Q_before_prep - 1, t)
                        prep_occupancy += 1
                        prep_time = exp_sample(rng, params.mean_prep)
                        schedule(t + prep_time, 'PREP_COMPLETE')
                    record_OR_change('BUSY', t)
                    surg_time = sample_surg_time(rng, params)
                    schedule(t + surg_time, 'SURG_COMPLETE')
                # else OR stays idle

    # ====================
    # Finalise statistics
    # ====================
    obs_time = params.run_length

    # Flush queue area
    start = max(last_Q_change, warmup)
    if end_time > start:
        area_Q += Q_before_prep * (end_time - start)
    # Flush preparation occupancy
    start = max(last_prep_change, warmup)
    if end_time > start:
        area_prep_busy += prep_occupancy * (end_time - start)

    # Flush OR state times
    start = max(last_OR_change, warmup)
    if end_time > start:
        dt = end_time - start
        if OR_state == 'BUSY':
            time_OR_busy += dt
        elif OR_state == 'BLOCKED':
            time_OR_blocked += dt
        elif OR_state == 'IDLE':
            time_OR_idle += dt

    # Flush recovery-full time
    start = max(last_rec_change, warmup)
    if end_time > start and rec_occupancy == params.R:
        time_rec_full += (end_time - start)

    avg_Q = area_Q / obs_time
    prob_OR_blocked = time_OR_blocked / obs_time
    prob_rec_full = time_rec_full / obs_time
    avg_prep_busy = area_prep_busy / obs_time
    prep_idle_fraction = 1.0 - (avg_prep_busy / params.P)
    OR_util = (time_OR_busy + time_OR_blocked) / obs_time

    return {
        "avg_Q": avg_Q,
        "prep_idle_frac": prep_idle_fraction,
        "prob_OR_blocked": prob_OR_blocked,
        "prob_rec_full": prob_rec_full,
        "OR_util": OR_util
    }


# ======================================
# 3. Confidence Interval (CI) utilities
# ======================================

def mean_ci(values, alpha=0.05):
    # Return (mean, lower, upper) 95% CI for a sample.
    n = len(values)
    mean = sum(values) / n
    if n < 2:
        return mean, None, None
    var = sum((x - mean) ** 2 for x in values) / (n - 1)
    s = math.sqrt(var)
    # For n = 20, df = 19 => t ≈ 2.093
    t = 2.093 if n == 20 else 1.96
    halfwidth = t * s / math.sqrt(n)
    return mean, mean - halfwidth, mean + halfwidth


def independent_diff_ci(vals1, vals2):
    # CI for difference of means: vals1 - vals2 (independent samples).
    n1, n2 = len(vals1), len(vals2)
    m1, m2 = sum(vals1) / n1, sum(vals2) / n2
    v1 = sum((x - m1) ** 2 for x in vals1) / (n1 - 1)
    v2 = sum((x - m2) ** 2 for x in vals2) / (n2 - 1)
    diff = m1 - m2
    se = math.sqrt(v1 / n1 + v2 / n2)
    t = 2.093 if min(n1, n2) == 20 else 1.96
    halfwidth = t * se
    return diff, diff - halfwidth, diff + halfwidth


def paired_diff_ci(vals1, vals2):
    # CI for paired differences: mean(vals1 - vals2).
    diffs = [x - y for x, y in zip(vals1, vals2)]
    return mean_ci(diffs)


def run_reps(params: SimParams, nrep=20, base_seed=1000):
    # Run nrep replications with different seeds (common seeds across configurations for pairing).
    metrics = {"avg_Q": [], "prep_idle_frac": [], "prob_OR_blocked": [], "prob_rec_full": [], "OR_util": []}
    for i in range(nrep):
        seed = base_seed + i
        res = simulate(params, seed)
        for k in metrics:
            metrics[k].append(res[k])
    return metrics


# ======================
# 4. Main experiments
# ======================

def print_summary(name, metrics):
    print(f"\nConfiguration: {name}")
    for metric, vals in metrics.items():
        m, lo, hi = mean_ci(vals)
        print(f"  {metric:17s}: mean={m:.3f}, 95% CI=({lo:.3f}, {hi:.3f})")


def main():
    print("\n ****** Base configurations ******")
    configs = {
        "3p4r": SimParams(P=3, R=4),
        "3p5r": SimParams(P=3, R=5),
        "4p5r": SimParams(P=4, R=5),
    }

    all_results = {}
    for name, p in configs.items():
        all_results[name] = run_reps(p, nrep=20, base_seed=1000)
        print_summary(name, all_results[name])

    # ============
    # Differences
    # ============
    print("\n ****** Independent differences (3p5r vs 4p5r, 3p5r vs 3p4r) ******")
    pairs = [("3p5r", "4p5r"), ("3p5r", "3p4r")]
    for a, b in pairs:
        print(f"\nIndependent: {a} - {b}")
        for metric in all_results[a]:
            d, lo, hi = independent_diff_ci(all_results[a][metric], all_results[b][metric])
            print(f"  {metric:17s}: diff={d:.3f}, 95% CI=({lo:.3f}, {hi:.3f})")

    print("\n****** Paired differences (common seeds per replication) ===")
    # Already used same seeds per replication index for each config.
    for a, b in pairs:
        print(f"\nPaired: {a} - {b}")
        for metric in all_results[a]:
            d, lo, hi = paired_diff_ci(all_results[a][metric], all_results[b][metric])
            print(f"  {metric:17s}: diff={d:.3f}, 95% CI=({lo:.3f}, {hi:.3f})")

    # ===========================================
    # Relative CI widths for blocking vs rec_full
    # ===========================================
    print("\n****** Relative CI widths for P(OR blocked) vs P(all recovery busy) ******")
    for name, res in all_results.items():
        print(f"\nConfiguration: {name}")
        for metric in ["prob_OR_blocked", "prob_rec_full"]:
            m, lo, hi = mean_ci(res[metric])
            hw = hi - m
            rel = hw / m if m > 0 else float('nan')
            print(f"  {metric:17s}: mean={m:.4f}, half-width={hw:.4f}, relative={rel:.3f}")

    # =======================
    # Twisted scenario (3p5r)
    # =======================
    print("\n****** Twisted scenario for 3p5r (same mean OR time, higher variance) ******")
    original = all_results["3p5r"]
    twisted_params = SimParams(P=3, R=5, twisted=True)
    twisted = run_reps(twisted_params, nrep=20, base_seed=1000)

    print("\nOriginal 3p5r:")
    print_summary("3p5r original", original)
    print("\nTwisted 3p5r:")
    print_summary("3p5r twisted", twisted)

    print("\nPaired differences: twisted - original (3p5r)")
    for metric in original:
        d, lo, hi = paired_diff_ci(twisted[metric], original[metric])
        print(f"  {metric:17s}: diff={d:.3f}, 95% CI=({lo:.3f}, {hi:.3f})")


if __name__ == "__main__":
    main()



 ****** Base configurations ******

Configuration: 3p4r
  avg_Q            : mean=3.201, 95% CI=(1.350, 5.053)
  prep_idle_frac   : mean=0.256, 95% CI=(0.160, 0.352)
  prob_OR_blocked  : mean=0.025, 95% CI=(0.009, 0.040)
  prob_rec_full    : mean=0.071, 95% CI=(0.048, 0.094)
  OR_util          : mean=0.795, 95% CI=(0.741, 0.849)

Configuration: 3p5r
  avg_Q            : mean=3.396, 95% CI=(1.656, 5.135)
  prep_idle_frac   : mean=0.254, 95% CI=(0.152, 0.357)
  prob_OR_blocked  : mean=0.004, 95% CI=(-0.001, 0.008)
  prob_rec_full    : mean=0.017, 95% CI=(0.007, 0.027)
  OR_util          : mean=0.803, 95% CI=(0.745, 0.860)

Configuration: 4p5r
  avg_Q            : mean=0.935, 95% CI=(0.558, 1.313)
  prep_idle_frac   : mean=0.452, 95% CI=(0.386, 0.519)
  prob_OR_blocked  : mean=0.007, 95% CI=(0.001, 0.013)
  prob_rec_full    : mean=0.021, 95% CI=(0.007, 0.035)
  OR_util          : mean=0.789, 95% CI=(0.729, 0.849)

 ****** Independent differences (3p5r vs 4p5r, 3p5r vs 3p4r) ******

Indep